In [1]:
from pathlib import Path
import re
import json
from collections import Counter

TRACE_DIR = Path("traces")
OUT_JSONL = Path("satp_path_summaries.jsonl")
OUT_TXT = Path("satp_path_summaries.txt")
PROMPT_DIR = Path("prompts")
PROMPT_DIR.mkdir(parents=True, exist_ok=True)

def split_trace_blocks(text):
    lines = text.splitlines()
    starts = [i for i, line in enumerate(lines) if line.strip().startswith("(trace")]
    if not starts:
        return [text] if text.strip() else []

    blocks = []
    for i, start in enumerate(starts):
        end = starts[i + 1] if i + 1 < len(starts) else len(lines)
        block = "\n".join(lines[start:end]).strip()
        if block:
            blocks.append(block)
    return blocks

def get_privilege(filename):
    if filename.startswith("Machine_"):
        return "Machine"
    if filename.startswith("Supervisor_"):
        return "Supervisor"
    if filename.startswith("User_"):
        return "User"
    return "Unknown"

def infer_instruction(filename):
    name = filename
    for p in ["Machine_", "Supervisor_", "User_"]:
        if name.startswith(p):
            name = name[len(p):]
    name = name.removesuffix(".trace")
    name = name.replace("_____", " ")
    name = name.replace("__", " ")
    name = name.replace("_", " ")
    return re.sub(r"\s+", " ", name).strip()

def has_mcause_2(block):
    return bool(re.search(r"write-reg \|mcause\|.*#x0*2\)", block, re.S))

def has_mcause_22(block):
    return bool(re.search(r"write-reg \|mcause\|.*#x0*16\)", block, re.S))

def exception_name(block):
    if has_mcause_22(block):
        return "virtual instruction"
    if has_mcause_2(block):
        return "illegal instruction"
    if "write-reg |mcause|" in block:
        return "exception"
    return "none"

def has_event(block, event):
    return event in block

def important_events(block):
    keywords = [
        "assume-reg |cur_privilege|",
        "assume-reg |satp|",
        "read-reg |cur_privilege|",
        "write-reg |cur_privilege|",
        "read-reg |mstatus|",
        "write-reg |mstatus|",
        "read-reg |satp|",
        "write-reg |satp|",
        "read-reg |misa|",
        "read-reg |x10|",
        "read-reg |x11|",
        "write-reg |x10|",
        "read-reg |mcause|",
        "write-reg |mcause|",
        "write-reg |mtval|",
        "write-reg |mepc|",
        "write-reg |nextPC|",
        "Retired",
    ]

    out = []
    for line in block.splitlines():
        s = line.strip()
        if any(k in s for k in keywords):
            out.append(s)
    return out[:120]

def instruction_kind(instr):
    low = instr.lower()

    if low.startswith("csrr "):
        return "read"
    if low.startswith("csrw "):
        return "write"
    if low.startswith("csrrw "):
        return "read_write"
    if low.startswith("csrrs ") or low.startswith("csrrc "):
        return "read_modify_write"
    if low.startswith("csrrwi "):
        return "read_write_immediate"
    if low.startswith("csrrsi ") or low.startswith("csrrci "):
        return "read_modify_write_immediate"

    return "unknown"

def infer_behavior(priv, instr, block):
    exc = exception_name(block)
    kind = instruction_kind(instr)

    reads_satp = has_event(block, "read-reg |satp|")
    writes_satp = has_event(block, "write-reg |satp|")
    reads_mstatus = has_event(block, "read-reg |mstatus|")
    writes_x10 = has_event(block, "write-reg |x10|")
    reads_x11 = has_event(block, "read-reg |x11|")
    reads_cur_priv = has_event(block, "read-reg |cur_privilege|")

    if exc == "illegal instruction":
        if priv == "User":
            name = "User-mode SATP access illegal"
            summary = "SATP CSR access from User mode raises an illegal instruction exception; SATP is not read or written normally."
        elif priv == "Supervisor" and reads_mstatus:
            name = "Supervisor-mode SATP access blocked by TVM-related path"
            summary = "Supervisor-mode SATP access can raise an illegal instruction exception when the SATP accessibility check fails through mstatus.TVM."
        else:
            name = f"{priv}-mode SATP access illegal"
            summary = f"{priv}-mode SATP access raises an illegal instruction exception on this trace path."
    elif exc == "virtual instruction":
        name = f"{priv}-mode SATP access virtual instruction"
        summary = f"{priv}-mode SATP access raises a virtual instruction exception on this trace path."
    else:
        if writes_satp and writes_x10 and reads_satp:
            name = f"{priv}-mode SATP read/write succeeds"
            summary = f"{priv}-mode SATP CSR operation succeeds: SATP is read, SATP may be written through the legalization path, and the old SATP value is written to x10."
        elif writes_satp:
            name = f"{priv}-mode SATP write succeeds"
            summary = f"{priv}-mode SATP write succeeds: the new SATP value is accepted by the Sail path and written to SATP."
        elif reads_satp and writes_x10:
            name = f"{priv}-mode SATP read succeeds"
            summary = f"{priv}-mode SATP read succeeds and writes the SATP value to x10."
        else:
            name = f"{priv}-mode SATP path succeeds"
            summary = f"{priv}-mode SATP-related path completes without an exception."

    return {
        "path_name": name,
        "summary": summary,
        "exception": exc,
        "instruction_kind": kind,
        "reads_satp": reads_satp,
        "writes_satp": writes_satp,
        "reads_mstatus": reads_mstatus,
        "reads_cur_privilege": reads_cur_priv,
        "reads_x11": reads_x11,
        "writes_x10": writes_x10,
    }

rows = []

for trace_file in sorted(TRACE_DIR.glob("*.trace")):
    text = trace_file.read_text(errors="replace")
    blocks = split_trace_blocks(text)

    for idx, block in enumerate(blocks):
        instr = infer_instruction(trace_file.name)
        priv = get_privilege(trace_file.name)

        if "satp" not in instr.lower() and "|satp|" not in block:
            continue

        info = infer_behavior(priv, instr, block)
        events = important_events(block)

        row = {
            "path_id": f"{trace_file.stem}__path_{idx:03d}",
            "trace_file": str(trace_file),
            "privilege": priv,
            "instruction": instr,
            **info,
            "important_trace_events": events,
        }

        rows.append(row)

with OUT_JSONL.open("w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

lines = []

for i, row in enumerate(rows, 1):
    lines.append(f"Path {i}: {row['path_name']}")
    lines.append(f"Path ID: {row['path_id']}")
    lines.append(f"Trace file: {row['trace_file']}")
    lines.append("")
    lines.append("Condition:")
    lines.append(f"- Privilege: {row['privilege']}.")
    lines.append(f"- Instruction: {row['instruction']}.")
    lines.append(f"- Instruction kind: {row['instruction_kind']}.")
    lines.append(f"- Reads cur_privilege: {row['reads_cur_privilege']}.")
    lines.append(f"- Reads mstatus: {row['reads_mstatus']}.")
    lines.append(f"- Reads satp: {row['reads_satp']}.")
    lines.append(f"- Writes satp: {row['writes_satp']}.")
    lines.append(f"- Reads x11: {row['reads_x11']}.")
    lines.append("")
    lines.append("Outcome:")
    lines.append(f"- Exception: {row['exception']}.")
    if row["exception"] == "none":
        lines.append("- SATP access: allowed.")
        if row["writes_satp"]:
            lines.append("- SATP write/update: yes.")
        else:
            lines.append("- SATP write/update: no.")
        if row["writes_x10"]:
            lines.append("- Output register x10 is written.")
    else:
        lines.append("- SATP access: denied/trapped before normal completion.")
        lines.append("- Normal SATP update: no.")
    lines.append("")
    lines.append("Summary:")
    lines.append(row["summary"])
    lines.append("")
    lines.append("Important trace events:")
    for e in row["important_trace_events"]:
        lines.append(f"- {e}")
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

OUT_TXT.write_text("\n".join(lines), encoding="utf-8")

for row in rows:
    prompt = f"""You are given one symbolic execution trace-derived path from Sail/Isla for RISC-V.

Task:
Convert this path into exactly one atomic architectural rule about SATP.

Important:
- Use only the path summary and trace events.
- Do not use emulator behavior.
- Do not use known bugs.
- Do not infer unrelated behavior.
- The target CSR/register is satp.
- The rule must describe one condition -> architectural outcome.
- Output only one line beginning with RULE:.

Path summary:
{json.dumps({k: v for k, v in row.items() if k != "important_trace_events"}, indent=2)}

Important trace events:
{chr(10).join(row["important_trace_events"])}
"""
    (PROMPT_DIR / f"{row['path_id']}.txt").write_text(prompt, encoding="utf-8")

print("trace files:", len(list(TRACE_DIR.glob("*.trace"))))
print("trace-derived paths:", len(rows))
print("unique summaries:", len(set(r["summary"] for r in rows)))
print("exception counts:")
for k, v in Counter(r["exception"] for r in rows).items():
    print(f"  {k}: {v}")
print("wrote:", OUT_JSONL)
print("wrote:", OUT_TXT)
print("wrote prompts:", PROMPT_DIR)


trace files: 27
trace-derived paths: 54
unique summaries: 8
exception counts:
  none: 36
  illegal instruction: 18
wrote: satp_path_summaries.jsonl
wrote: satp_path_summaries.txt
wrote prompts: prompts


In [2]:
import json
from pathlib import Path
from collections import Counter

rows = [json.loads(x) for x in Path("satp_path_summaries.jsonl").read_text().splitlines() if x.strip()]

print("paths:", len(rows))
print("unique path ids:", len(set(r["path_id"] for r in rows)))
print("exception:", Counter(r["exception"] for r in rows))
print("privilege:", Counter(r["privilege"] for r in rows))
print("summary:")
for k, v in Counter(r["summary"] for r in rows).most_common():
    print(v, "-", k)

paths: 54
unique path ids: 54
exception: Counter({'none': 36, 'illegal instruction': 18})
privilege: Counter({'Supervisor': 27, 'Machine': 18, 'User': 9})
summary:
13 - Machine-mode SATP CSR operation succeeds: SATP is read, SATP may be written through the legalization path, and the old SATP value is written to x10.
13 - Supervisor-mode SATP CSR operation succeeds: SATP is read, SATP may be written through the legalization path, and the old SATP value is written to x10.
9 - Supervisor-mode SATP access can raise an illegal instruction exception when the SATP accessibility check fails through mstatus.TVM.
9 - SATP CSR access from User mode raises an illegal instruction exception; SATP is not read or written normally.
4 - Machine-mode SATP write succeeds: the new SATP value is accepted by the Sail path and written to SATP.
4 - Supervisor-mode SATP write succeeds: the new SATP value is accepted by the Sail path and written to SATP.
1 - Machine-mode SATP read succeeds and writes the SATP va